In [2]:
import pandas as pd
import numpy as np
import logging
import json
from pathlib import Path
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# ================== CONFIG ==================
np.random.seed(42)

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

RAW_PATH = Path('dataset/raw/')
PROCESSED_PATH = Path('dataset/processed/')
PROCESSED_PATH.mkdir(parents=True, exist_ok=True)

# ===== DATA QUALITY REPORT (industry practice) =====
data_quality_report = {}

def log_quality(dataset_name: str, df: pd.DataFrame, stage: str):
    """Track data quality at each stage — interviewers love this."""
    report = {
        'stage': stage,
        'rows': int(df.shape[0]),
        'columns': int(df.shape[1]),
        'missing_pct': round(df.isnull().mean().mean() * 100, 2),
        'duplicates': int(df.duplicated().sum()),
        'memory_mb': round(df.memory_usage(deep=True).sum() / 1e6, 2),
        'timestamp': datetime.now().isoformat()
    }
    data_quality_report[f"{dataset_name}_{stage}"] = report
    logger.info(f"[{dataset_name}] {stage}: {report['rows']:,} rows, "
                f"{report['missing_pct']}% missing, "
                f"{report['memory_mb']}MB")
    return report


def load_and_validate(file_name: str, required_cols: list = None):
    """Load CSV with validation and error handling."""
    try:
        file_path = RAW_PATH / file_name

        df = pd.read_csv(
            file_path,
            low_memory=False,
            encoding="latin1",
            on_bad_lines="skip"
        )

        logger.info(f"Loaded {file_name} -> Shape: {df.shape}")
        log_quality(file_name, df, 'raw')

        if required_cols:
            missing = [c for c in required_cols if c not in df.columns]
            if missing:
                raise ValueError(f"Missing columns in {file_name}: {missing}")

        return df

    except Exception as e:
        logger.error(f"Failed to load {file_name}: {e}")
        raise


# ================== LOAD ALL DATASETS ==================
zomato = load_and_validate('global_zomato_restaurants.csv')
eta = load_and_validate('food_delivery_eta.csv')
orders = load_and_validate('order_history.csv')


# ====================== ZOMATO CLEANING ======================
logger.info("=" * 50)
logger.info("CLEANING ZOMATO DATASET")
logger.info("=" * 50)

zomato.columns = (zomato.columns
                   .str.strip()
                   .str.lower()
                   .str.replace(' ', '_')    # use underscore, NOT remove
                   .str.replace('-', '_'))

# Deduplicate
before = len(zomato)
zomato = zomato.drop_duplicates(subset=['restaurant_id'], keep='first')
logger.info(f"Removed {before - len(zomato)} duplicate restaurants")

# Type casting + cleaning
zomato['cuisines'] = (zomato['cuisines']
                      .fillna('Unknown')
                      .str.lower()
                      .str.strip())

zomato['price_range'] = (pd.to_numeric(zomato['price_range'], errors='coerce')
                         .fillna(2)
                         .astype(int)
                         .clip(1, 4))

zomato['average_cost_for_two'] = (pd.to_numeric(
                                      zomato['average_cost_for_two'],
                                      errors='coerce')
                                  .clip(lower=50, upper=10000))

zomato['aggregate_rating'] = (pd.to_numeric(
                                  zomato['aggregate_rating'],
                                  errors='coerce')
                              .clip(0, 5)
                              .fillna(3.5))

zomato['votes'] = (pd.to_numeric(zomato['votes'], errors='coerce')
                   .fillna(0)
                   .astype(int))

# ===== NEW: Primary cuisine extraction (first cuisine only) =====
zomato['primary_cuisine'] = (zomato['cuisines']
                             .str.split(',')
                             .str[0]
                             .str.strip())

log_quality('zomato', zomato, 'cleaned')
zomato.to_csv(PROCESSED_PATH / 'zomato_cleaned.csv', index=False)
logger.info(f"Zomato saved -> {zomato.shape}")


# ====================== ETA CLEANING ======================
logger.info("=" * 50)
logger.info("CLEANING ETA DATASET")
logger.info("=" * 50)

eta.columns = (eta.columns
               .str.strip()
               .str.lower()
               .str.replace(' ', '_'))

# ===== FIX: Safe column detection for prep time =====
prep_col = None
for candidate in ['preparation_time_min', 'preparatory_time_min',
                   'prep_time_min', 'prep_time']:
    if candidate in eta.columns:
        prep_col = candidate
        break

if prep_col:
    eta['preparation_time_min'] = (pd.to_numeric(eta[prep_col], errors='coerce')
                                   .clip(5, 60)
                                   .fillna(15))
    if prep_col != 'preparation_time_min':
        eta = eta.drop(columns=[prep_col])
else:
    logger.warning("No prep time column found — generating synthetic")
    eta['preparation_time_min'] = np.random.randint(8, 35, size=len(eta))

# Distance + delivery time
eta['distance_km'] = (pd.to_numeric(eta['distance_km'], errors='coerce')
                      .clip(lower=0.5, upper=50)
                      .fillna(5.0))

eta['delivery_time_min'] = (pd.to_numeric(eta['delivery_time_min'], errors='coerce')
                            .clip(10, 120)
                            .fillna(30))

# Categorical cleaning
eta['weather'] = eta['weather'].fillna('clear').str.lower().str.strip()
eta['traffic_level'] = eta['traffic_level'].fillna('medium').str.lower().str.strip()
eta['time_of_day'] = eta['time_of_day'].fillna('evening').str.lower().str.strip()

# ===== NEW: Validate delivery_time >= prep_time (logical check) =====
invalid_mask = eta['delivery_time_min'] < eta['preparation_time_min']
logger.info(f"Fixing {invalid_mask.sum()} rows where delivery < prep time")
eta.loc[invalid_mask, 'delivery_time_min'] = (
    eta.loc[invalid_mask, 'preparation_time_min'] + 
    np.random.randint(5, 20, size=invalid_mask.sum())
)

log_quality('eta', eta, 'cleaned')
eta.to_csv(PROCESSED_PATH / 'eta_cleaned.csv', index=False)
logger.info(f"ETA saved -> {eta.shape}")


# ====================== DEMAND PREPARATION ======================
logger.info("=" * 50)
logger.info("BUILDING DEMAND DATASET")
logger.info("=" * 50)

orders.columns = (orders.columns
                  .str.strip()
                  .str.lower()
                  .str.replace(' ', '_'))

orders['order_time'] = pd.to_datetime(orders['order_time'], errors='coerce')
before = len(orders)
orders = orders.dropna(subset=['order_time', 'restaurant_id'])
logger.info(f"Dropped {before - len(orders)} rows with null time/restaurant")

# ===== FIX: Extract integer hour BEFORE groupby =====
orders['hour_of_day'] = orders['order_time'].dt.hour
orders['day_of_week'] = orders['order_time'].dt.dayofweek
orders['date'] = orders['order_time'].dt.date

# Hourly aggregation (FIXED — clean integer columns)
demand = (orders
          .groupby(['restaurant_id', 'date', 'hour_of_day', 'day_of_week'])
          .agg(order_count=('order_id', 'count'))      # FIX: renamed to order_count
          .reset_index())

demand['is_weekend'] = (demand['day_of_week'] >= 5).astype(int)
demand['holiday'] = 0  # FIX: add holiday column to real data too

# Merge restaurant features
zomato_features = zomato[['restaurant_id', 'aggregate_rating', 'price_range']].copy()
demand = demand.merge(zomato_features, on='restaurant_id', how='left')
demand['restaurant_rating'] = demand['aggregate_rating'].fillna(3.8)
demand['price_range'] = demand['price_range'].fillna(2).astype(int)
demand = demand.drop(columns=['aggregate_rating', 'date'], errors='ignore')

logger.info(f"Real demand rows: {len(demand):,}")


# ===== SYNTHETIC AUGMENTATION (improved) =====
# Constants (not magic numbers)
PEAK_LUNCH = [12, 13]
PEAK_DINNER = [19, 20, 21]
BASE_ORDERS = 35
PEAK_BOOST = 45
WEEKEND_BOOST = 25
HOLIDAY_PROB = 0.08
NUM_RESTAURANTS = 150

rest_ids = demand['restaurant_id'].unique()[:NUM_RESTAURANTS]
if len(rest_ids) < NUM_RESTAURANTS:
    # If not enough real restaurants, generate IDs
    extra = [f"SYN_{i}" for i in range(NUM_RESTAURANTS - len(rest_ids))]
    rest_ids = np.concatenate([rest_ids, extra])

synthetic_rows = []
for rest in rest_ids:
    rest_rating = round(np.random.uniform(3.2, 4.9), 1)
    rest_price = np.random.randint(1, 5)
    for d in range(7):
        for h in range(24):
            base = BASE_ORDERS + np.random.normal(0, 12)
            if h in PEAK_LUNCH + PEAK_DINNER:
                base += PEAK_BOOST
            if d >= 5:
                base += WEEKEND_BOOST
            if h < 6:
                base *= 0.3  # NEW: dead hours (realistic)

            is_holiday = 1 if np.random.random() < HOLIDAY_PROB else 0
            if is_holiday:
                base *= 1.4  # NEW: holiday multiplier

            synthetic_rows.append({
                'restaurant_id': rest,
                'day_of_week': d,
                'hour_of_day': h,
                'is_weekend': 1 if d >= 5 else 0,
                'holiday': is_holiday,
                'restaurant_rating': rest_rating,
                'price_range': rest_price,
                'order_count': max(3, int(base))  # FIX: same name as real data
            })

syn_df = pd.DataFrame(synthetic_rows)
logger.info(f"Synthetic rows generated: {len(syn_df):,}")

# ===== FIX: Align columns before concat =====
common_cols = ['restaurant_id', 'day_of_week', 'hour_of_day',
               'is_weekend', 'holiday', 'restaurant_rating',
               'price_range', 'order_count']

demand_aligned = demand.reindex(columns=common_cols)
syn_aligned = syn_df.reindex(columns=common_cols)

final_demand = pd.concat([demand_aligned, syn_aligned], ignore_index=True)

# ===== NEW: Add source column (track real vs synthetic) =====
final_demand['data_source'] = (['real'] * len(demand_aligned) + 
                                ['synthetic'] * len(syn_aligned))

# ===== NEW: Lag features (critical for time-series) =====
final_demand = final_demand.sort_values(
    ['restaurant_id', 'day_of_week', 'hour_of_day']
).reset_index(drop=True)

final_demand['lag_1h'] = (final_demand
                          .groupby('restaurant_id')['order_count']
                          .shift(1)
                          .fillna(method='bfill'))

final_demand['rolling_3h_avg'] = (final_demand
                                   .groupby('restaurant_id')['order_count']
                                   .transform(
                                       lambda x: x.rolling(3, min_periods=1).mean()
                                   ))

# Final validation
assert final_demand.isnull().sum().sum() == 0, "NaN found in final demand!"
assert 'order_count' in final_demand.columns, "Target column missing!"

log_quality('demand', final_demand, 'final')
final_demand.to_csv(PROCESSED_PATH / 'demand_train.csv', index=False)
logger.info(f"Final demand saved -> {final_demand.shape}")
print(f"\nFinal columns: {final_demand.columns.tolist()}")
print(final_demand.head())
print(f"\norder_count stats:\n{final_demand['order_count'].describe()}")


# ===== SAVE DATA QUALITY REPORT =====
with open(PROCESSED_PATH / 'data_quality_report.json', 'w') as f:
    json.dump(data_quality_report, f, indent=2)
logger.info("Data quality report saved")


# ===== FINAL SUMMARY =====
print("\n" + "=" * 60)
print("PHASE 1 COMPLETE — ALL DATASETS CLEANED")
print("=" * 60)
print(f"  Zomato restaurants : {len(zomato):>8,} rows")
print(f"  ETA delivery       : {len(eta):>8,} rows")
print(f"  Demand (final)     : {len(final_demand):>8,} rows")
print(f"  Quality report     : saved to {PROCESSED_PATH / 'data_quality_report.json'}")
print(f"  Files in processed : {[f.name for f in PROCESSED_PATH.glob('*.csv')]}")
print("=" * 60)

2026-03-16 17:47:09,755 - INFO - Loaded global_zomato_restaurants.csv -> Shape: (9551, 18)
2026-03-16 17:47:09,851 - INFO - [global_zomato_restaurants.csv] raw: 9,551 rows, 0.01% missing, 7.11MB
2026-03-16 17:47:09,871 - INFO - Loaded food_delivery_eta.csv -> Shape: (1000, 9)
2026-03-16 17:47:09,876 - INFO - [food_delivery_eta.csv] raw: 1,000 rows, 1.33% missing, 0.26MB
2026-03-16 17:47:10,274 - INFO - Loaded order_history.csv -> Shape: (100000, 23)
2026-03-16 17:47:10,628 - INFO - [order_history.csv] raw: 100,000 rows, 0.0% missing, 68.28MB
2026-03-16 17:47:10,629 - INFO - ==================================================
2026-03-16 17:47:10,631 - INFO - CLEANING ZOMATO DATASET
2026-03-16 17:47:10,632 - INFO - ==================================================
2026-03-16 17:47:10,637 - INFO - Removed 0 duplicate restaurants
2026-03-16 17:47:10,701 - INFO - [zomato] cleaned: 9,551 rows, 0.0% missing, 7.66MB
2026-03-16 17:47:10,792 - INFO - Zomato saved -> (9551, 19)
2026-03-16 17:47:1


Final columns: ['restaurant_id', 'day_of_week', 'hour_of_day', 'is_weekend', 'holiday', 'restaurant_rating', 'price_range', 'order_count', 'data_source', 'lag_1h', 'rolling_3h_avg']
   restaurant_id  day_of_week  hour_of_day  is_weekend  holiday  \
0              1            0            0           0        0   
1              1            0            0           0        0   
2              1            0            1           0        0   
3              1            0            1           0        0   
4              1            0            2           0        0   

   restaurant_rating  price_range  order_count data_source  lag_1h  \
0                3.8            2            1        real     1.0   
1                3.8            1           12   synthetic     1.0   
2                3.8            2            1        real    12.0   
3                3.8            1            8   synthetic     1.0   
4                3.8            2            1        real     8